In [247]:
import os
import re
import random
from collections import defaultdict, Counter

In [248]:
folder = "corpus.viwiki/viwiki"
corpus = []
files = [f for f in os.listdir(folder) if f.endswith(".txt")]

for fname in files[:2000]:
    fpath = os.path.join(folder, fname)
    with open(fpath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("=="):
                continue
            for sent in re.split(r"[.!?\n]+", line):
                sent = sent.strip()
                if sent:
                    corpus.append(sent.lower())

print(len(corpus))

175411


In [249]:
def clean_token(token):
    return re.sub(r'[^\w\s]', '', token)

def tokenize(sentence: str):
    tokens = [clean_token(t) for t in sentence.split()]
    tokens = [t for t in tokens if t] 
    return ["<s>"] + tokens + ["</s>"]

In [250]:
def build_counts(corpus):
    unigram_counts: Counter = Counter()
    bigram_counts: Counter = Counter()

    for sentence in corpus:
        tokens = tokenize(sentence)
        for i in range(len(tokens) - 1):
            unigram_counts[tokens[i]] += 1
            bigram_counts[(tokens[i], tokens[i + 1])] += 1
        unigram_counts[tokens[-1]] += 1  #count <s>

    return unigram_counts, bigram_counts

In [251]:
def bigram_probability(w1, w2, unigram_counts, bigram_counts, vocab_size):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + vocab_size)

In [252]:
def sentence_probability(sentence, unigram_counts, bigram_counts, vocab_size):
    tokens = tokenize(sentence.lower())
    prob = 1.0
    for i in range(len(tokens) - 1):
        prob *= bigram_probability(tokens[i], tokens[i + 1], unigram_counts, bigram_counts, vocab_size)
    return prob


sentences = [
    "Hôm nay thời tiết tại Hà Nội khá đẹp",
    "Chính phủ ban hành nghị định mới về lao động",
    "Giá xăng trong nước tiếp tục tăng",
    "Nhiều người dân tham gia giao thông trong giờ cao điểm",
    "Đội tuyển Việt Nam giành chiến thắng quan trọng",
    "Kinh tế Việt Nam đang phục hồi mạnh mẽ",
    "Các chuyên gia dự báo thời tiết sẽ có mưa lớn",
    "Thành phố triển khai nhiều dự án giao thông mới",
    "Người dân đổ về trung tâm thành phố vào cuối tuần",
    "Nhiều doanh nghiệp đầu tư vào công nghệ mới"
]

for s in sentences:
    prob = sentence_probability(
        s,
        unigram_counts,
        bigram_counts,
        len(unigram_counts)
    )
    print("Sentence:", s)
    print("Probability:", prob)
    print()

Sentence: Hôm nay thời tiết tại Hà Nội khá đẹp
Probability: 1.6183020247123375e-36

Sentence: Chính phủ ban hành nghị định mới về lao động
Probability: 8.562185900246076e-32

Sentence: Giá xăng trong nước tiếp tục tăng
Probability: 1.243381055222987e-27

Sentence: Nhiều người dân tham gia giao thông trong giờ cao điểm
Probability: 4.051938630892402e-37

Sentence: Đội tuyển Việt Nam giành chiến thắng quan trọng
Probability: 1.2572499517583666e-27

Sentence: Kinh tế Việt Nam đang phục hồi mạnh mẽ
Probability: 1.0982088824231415e-30

Sentence: Các chuyên gia dự báo thời tiết sẽ có mưa lớn
Probability: 1.2277571653972228e-39

Sentence: Thành phố triển khai nhiều dự án giao thông mới
Probability: 1.0573723528790447e-35

Sentence: Người dân đổ về trung tâm thành phố vào cuối tuần
Probability: 2.3023690274714484e-36

Sentence: Nhiều doanh nghiệp đầu tư vào công nghệ mới
Probability: 2.4905166289660153e-30



In [253]:
def build_lookup(bigram_counts):
    lookup = defaultdict(list)
    for (w1, w2) in bigram_counts:
        lookup[w1].append(w2)
    return dict(lookup)

In [254]:
def generate_sentence(lookup, unigram_counts, bigram_counts, vocab_size, max_len = 20):
    current = "<s>"
    words = []

    for _ in range(max_len):
        candidates = lookup.get(current, [])
        if not candidates:
            break

        weights = [bigram_probability(current, c, unigram_counts, bigram_counts, vocab_size) for c in candidates]
        total = sum(weights)
        weights = [w / total for w in weights]
        chosen = random.choices(candidates, weights=weights, k=1)[0]

        if chosen == "</s>":
            break

        words.append(chosen)
        current = chosen

    if not words:
        return ""
    return " ".join(words).capitalize() + "."

lookup = build_lookup(bigram_counts)

for i in range(5):
    print(generate_sentence(
        lookup,
        unigram_counts,
        bigram_counts,
        len(unigram_counts)
    ))

Phi cơ cấu công nghiệp từ long trong 30 trong nhà nữ ngôn ngữ học nhà của trung quốc tế.
Hillary nhà thờ ấn hàng loại tiêm ở phần còn dưới đây được truy cập nhật tại các tổ chức.
Theo cách 2 lần thứ nhất trong bài cộng hòa trên 2 phường hà nội các dạng cây vợt khi.
Numismatist in ethics ông qua thành clb thể bị đập bandiamir đã từng tham gia.
19 có một phần của ustaše bắt đầu ghi nhận cdma hiện là yêu cầu và đã chứng cho đảng.
